## BTreeMap Null Separated Keys
---

In [2]:
use std::{
	collections::{BTreeMap, HashMap},
	iter,
};

In [3]:
trait NullSeparatedKeys {
	fn get_tuples(&self, primary_key: &str, secondary_key: &str) -> Vec<(String, String)>;
    fn put_tuples(&mut self, primary_key: &str, tuples: Vec<(String, String)>);
}

impl NullSeparatedKeys for BTreeMap<String, String> {
	fn get_tuples(&self, primary_key: &str, secondary_key: &str) -> Vec<(String, String)> {
		let range_start = format!("{}\x00{}\x00", primary_key, secondary_key);
		let range_end = format!("{}\x00{}\x01", primary_key, secondary_key);
		self.range(range_start..range_end)
			.map(|(k, v)| (k.split("\x00").nth(2).unwrap_or("").to_string(), v.clone()))
			.chain(iter::once(("secondary_key".to_string(), secondary_key.to_string())))
			.collect::<Vec<(String, String)>>()
	}

    fn put_tuples(&mut self, primary_key: &str, tuples: Vec<(String, String)>) {
        let secondary_key = &tuples.iter().find(|(k,v)| k == "secondary_key").unwrap().1.clone();
        for tuple in tuples {
            if tuple.1 != "secondary_key" {
                self.insert(format!("{}\x00{}\x00{}", primary_key, secondary_key, tuple.0), tuple.1);
            }
        }
    }
}

In [4]:
#[derive(Debug)]
struct PasswordEntry {
	title: String,
	username: String,
	password: String,
	note: String,
	tags: String,
	url: String,
	expires_ts: String,
}

In [5]:
impl FromIterator<(String, String)> for PasswordEntry {
	fn from_iter<I: IntoIterator<Item = (String, String)>>(iter: I) -> Self {
		let mut map: HashMap<_, _> = iter.into_iter().collect();
		Self {
			title: map.remove("secondary_key").unwrap_or_default(),
			username: map.remove("username").unwrap_or_default(),
			password: map.remove("password").unwrap_or_default(),
			note: map.remove("note").unwrap_or_default(),
			tags: map.remove("tags").unwrap_or_default(),
			url: map.remove("url").unwrap_or_default(),
			expires_ts: map.remove("expires_ts").unwrap_or_default(),
		}
	}
}

In [6]:
impl IntoIterator for PasswordEntry {
    type Item = (String, String);
    type IntoIter = std::vec::IntoIter<Self::Item>;

    fn into_iter(self) -> Self::IntoIter {
        vec![
            ("secondary_key".into(), self.title),
            ("username".into(), self.username),
            ("password".into(), self.password),
            ("note".into(), self.note),
            ("tags".into(), self.tags),
            ("url".into(), self.url),
            ("expires_ts".into(), self.expires_ts),
        ].into_iter()
    }
}

In [7]:
let mut nskmap = BTreeMap::<String, String>::new();
nskmap.insert("password_entry\x00bank\x00username".into(), "myname".into());
nskmap.insert("password_entry\x00bank\x00password".into(), "opensesame".into());
nskmap.insert("password_entry\x00email\x00username".into(), "myname".into());
nskmap.insert("password_entry\x00email\x00password".into(), "123456".into());
nskmap

{"password_entry\0bank\0password": "opensesame", "password_entry\0bank\0username": "myname", "password_entry\0email\0password": "123456", "password_entry\0email\0username": "myname"}

In [8]:
let tuples = nskmap.get_tuples("password_entry", "bank");
tuples

[("password", "opensesame"), ("username", "myname"), ("secondary_key", "bank")]

In [9]:
let pwe = tuples.into_iter().collect::<PasswordEntry>();
pwe

PasswordEntry { title: "bank", username: "myname", password: "opensesame", note: "", tags: "", url: "", expires_ts: "" }

In [10]:
let tuples = pwe.into_iter().collect::<Vec<(String, String)>>();
tuples

[("secondary_key", "bank"), ("username", "myname"), ("password", "opensesame"), ("note", ""), ("tags", ""), ("url", ""), ("expires_ts", "")]

In [11]:
let mut nskmap = BTreeMap::<String, String>::new();
nskmap.put_tuples("password_entry", tuples);
nskmap

{"password_entry\0bank\0expires_ts": "", "password_entry\0bank\0note": "", "password_entry\0bank\0password": "opensesame", "password_entry\0bank\0secondary_key": "bank", "password_entry\0bank\0tags": "", "password_entry\0bank\0url": "", "password_entry\0bank\0username": "myname"}